## **Bagian 1: Setup Environment & Persiapan Data**

In [ ]:
# ============================================================
# SEL 2.1: Mount Drive, Cek GPU, dan Install Library
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("GPU TIDAK TERDETEKSI! Silakan ganti Runtime ke T4 GPU.")

# Install library yang dibutuhkan
!pip install timm --quiet
!pip install albumentations --quiet
!pip install scikit-learn --quiet

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import shutil
import time
import warnings
warnings.filterwarnings('ignore')

print("Semua library berhasil diimport.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch Version: 2.11.0+cu128
CUDA Available: True
GPU: Tesla T4
VRAM: 15.6 GB
Semua library berhasil diimport.


In [ ]:
# ============================================================
# SEL 2.2 (REVISI): Ekstrak Dataset Langsung ke Local SSD
# ============================================================
import os
import shutil

# 1. Bersihkan folder lokal dari percobaan sebelumnya yang mungkin terinterrupt
if os.path.exists('/content/ffe_dataset'):
    shutil.rmtree('/content/ffe_dataset')

ZIP_PATH = '/content/drive/MyDrive/tugas_akhir/data/raw/ffe_dataset/ffe_dataset.zip'
EXTRACT_DIR = '/content/temp_extract'

if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
os.makedirs(EXTRACT_DIR)

# 2. Ekstrak ZIP menggunakan command line (Jauh lebih cepat daripada shutil/Python)
print("Mengekstrak ZIP langsung ke local SSD (Metode Cepat)...")
!unzip -q "$ZIP_PATH" -d "$EXTRACT_DIR"
print("Ekstraksi selesai. Memverifikasi struktur folder...")

# 3. Verifikasi dan rapikan struktur folder (Mencegah nested folder)
subdirs = [d for d in os.listdir(EXTRACT_DIR) if os.path.isdir(os.path.join(EXTRACT_DIR, d))]

if len(subdirs) == 1 and (' - ' in subdirs[0] or 'Fresh' in subdirs[0]):
    # Kasus A: Tidak ada root folder di dalam ZIP, folder kelas langsung ada di EXTRACT_DIR
    os.rename(EXTRACT_DIR, '/content/ffe_dataset')
elif len(subdirs) == 1:
    # Kasus B: Ada satu root folder di dalam ZIP (misal: 'ffe_dataset'), pindahkan isinya
    root_folder = os.path.join(EXTRACT_DIR, subdirs[0])
    os.rename(root_folder, '/content/ffe_dataset')
    shutil.rmtree(EXTRACT_DIR)
else:
    # Kasus C: Banyak folder kelas langsung berada di EXTRACT_DIR
    os.rename(EXTRACT_DIR, '/content/ffe_dataset')

# 4. Verifikasi Akhir
class_folders = sorted([d for d in os.listdir('/content/ffe_dataset')
                        if os.path.isdir(os.path.join('/content/ffe_dataset', d))])

print(f"Dataset berhasil diekstrak ke /content/ffe_dataset")
print(f"Jumlah folder kelas terdeteksi: {len(class_folders)}")

# Pastikan path ini sesuai dengan yang diharapkan oleh sel berikutnya
DATASET_LOCAL_PATH = '/content/ffe_dataset'

Mengekstrak ZIP langsung ke local SSD (Metode Cepat)...
Ekstraksi selesai. Memverifikasi struktur folder...
Dataset berhasil diekstrak ke /content/ffe_dataset
Jumlah folder kelas terdeteksi: 24


In [ ]:
# ============================================================
# SEL 2.3: Konfigurasi dan Transformasi Albumentations
# ============================================================
BASE_DRIVE = '/content/drive/MyDrive/tugas_akhir'
CONFIG = {
    'base_drive': BASE_DRIVE,
    'n_folds': 5,
    'batch_size': 16,      # Sesuaikan ke 8 jika terjadi Out of Memory (OOM)
    'num_workers': 2,
    'epochs': 50,
    'lr': 1e-4,
    'weight_decay': 1e-2,
    'patience': 10,        # Early stopping patience
    'image_size': 224,
    'seed': 42
}

# Set seed untuk reproducibility
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

# Augmentasi untuk Training (berdasarkan desain eksperimen)
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.5),
    A.Resize(CONFIG['image_size'], CONFIG['image_size']),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# Transformasi untuk Validasi dan Test (HANYA resize dan normalisasi)
val_test_transform = A.Compose([
    A.Resize(CONFIG['image_size'], CONFIG['image_size']),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

print("Konfigurasi dan augmentasi berhasil dimuat.")

Konfigurasi dan augmentasi berhasil dimuat.


## **Bagian 2: Dataset dan DataLoader**

In [ ]:
# ============================================================
# SEL 2.4: Custom PyTorch Dataset
# ============================================================
from torch.utils.data import Dataset, DataLoader
from sklearn.utils.class_weight import compute_class_weight

class FishEyeDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Path di CSV merujuk ke /content/ffe_dataset/...
        img_path = row['file_path']

        img = cv2.imread(img_path)
        if img is None:
            # Fallback jika ada gambar corrupt (berupa tensor kosong)
            img = np.zeros((CONFIG['image_size'], CONFIG['image_size'], 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform:
            img = self.transform(image=img)['image']

        label = torch.tensor(row['label'], dtype=torch.long)
        return img, label

print("Class FishEyeDataset berhasil didefinisikan.")

Class FishEyeDataset berhasil didefinisikan.


## **Bagian 3: Training Loop dengan Mixed Precision**

In [ ]:
# ============================================================
# SEL 2.5: Fungsi Training dan Evaluasi (dengan AMP / FP16)
# ============================================================
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import accuracy_score

def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for images, labels in tqdm(loader, desc="  Training", leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        # Mixed Precision Forward Pass
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        # Mixed Precision Backward Pass
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="  Evaluating", leave=False):
            images, labels = images.to(device), labels.to(device)

            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc

print("Fungsi training dan evaluasi siap.")

Fungsi training dan evaluasi siap.


## **Bagian 4: Eksekusi 5-Fold Cross Validation**

In [ ]:
# ============================================================
# SEL 2.6: Loop Training 5-Fold (Fine-Tuning Swin-Tiny)
# ============================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logs_dir = os.path.join(BASE_DRIVE, 'results/logs/swin_tiny')
ckpt_dir = os.path.join(BASE_DRIVE, 'checkpoints/swin_tiny')
os.makedirs(logs_dir, exist_ok=True)
os.makedirs(ckpt_dir, exist_ok=True)

for fold in range(1, CONFIG['n_folds'] + 1):
    print(f"\n{'='*20} MEMULAI FOLD {fold} {'='*20}")

    # 1. Load CSV paths untuk fold ini
    fold_dir = os.path.join(BASE_DRIVE, f'data/processed/fold_{fold}')
    df_train = pd.read_csv(os.path.join(fold_dir, 'train_paths.csv'))
    df_val = pd.read_csv(os.path.join(fold_dir, 'val_paths.csv'))

    # 2. Buat Dataset dan DataLoader
    train_ds = FishEyeDataset(df_train, transform=train_transform)
    val_ds = FishEyeDataset(df_val, transform=val_test_transform)

    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'],
                              shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'],
                            shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=True)

    # 3. Hitung Class Weights untuk menangani imbalance ringan
    class_weights = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=df_train['label'])
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
    criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

    # 4. Inisialisasi Model Swin-Tiny
    model = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=3)
    model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])
    scaler = GradScaler() # Untuk Mixed Precision

    # 5. Training Loop dengan Early Stopping
    best_val_loss = float('inf')
    patience_counter = 0
    fold_logs = []

    for epoch in range(1, CONFIG['epochs'] + 1):
        print(f"Epoch {epoch}/{CONFIG['epochs']}")
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")

        fold_logs.append({
            'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
            'val_loss': val_loss, 'val_acc': val_acc, 'lr': optimizer.param_groups[0]['lr']
        })

        # Save best model & Early Stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_ckpt_path = os.path.join(ckpt_dir, f'fold_{fold}_best.pth')
            torch.save(model.state_dict(), best_ckpt_path)
            print(f"  >> Model terbaik disimpan (Val Loss: {val_loss:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= CONFIG['patience']:
                print(f"  >> Early stopping triggered pada epoch {epoch}.")
                break

    # Simpan log training fold ini
    pd.DataFrame(fold_logs).to_csv(os.path.join(logs_dir, f'fold_{fold}_training_log.csv'), index=False)
    print(f"Fold {fold} selesai. Log disimpan.")


==================== MEMULAI FOLD 1 ====================


model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch 1/50


  Train Loss: 1.0006 | Train Acc: 0.5034
  Val Loss:   0.9273 | Val Acc:   0.5619
  >> Model terbaik disimpan (Val Loss: 0.9273)
Epoch 2/50


  Train Loss: 0.8564 | Train Acc: 0.6009
  Val Loss:   0.8357 | Val Acc:   0.5932
  >> Model terbaik disimpan (Val Loss: 0.8357)
Epoch 3/50


... (101 outputs left out)

  Train Loss: 0.1135 | Train Acc: 0.9594
  Val Loss:   0.6271 | Val Acc:   0.7923
Epoch 20/50


  Train Loss: 0.1121 | Train Acc: 0.9591
  Val Loss:   0.7544 | Val Acc:   0.7738
Epoch 21/50


  Train Loss: 0.1083 | Train Acc: 0.9616
  Val Loss:   0.8151 | Val Acc:   0.7838
Epoch 22/50


  Train Loss: 0.0676 | Train Acc: 0.9822
  Val Loss:   0.7542 | Val Acc:   0.7966
  >> Early stopping triggered pada epoch 22.
Fold 5 selesai. Log disimpan.


## **Bagian 5: Ekstraksi Deep Features (768 Dimensi)**

In [ ]:
# ============================================================
# SEL 2.7: Ekstraksi Deep Features dari Model Terbaik
# ============================================================
from torch.utils.data import DataLoader

def extract_features(model, loader, device):
    model.eval()
    all_features = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="  Ekstraksi Fitur", leave=False):
            images = images.to(device)
            with autocast():
                # Swin-Tiny dengan num_classes=0 dan global_pool='avg'
                # akan langsung mereturn vektor fitur (Batch, 768)
                features = model(images)
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())

    return np.vstack(all_features), np.concatenate(all_labels)

feat_dir = os.path.join(BASE_DRIVE, 'data/features/deep')

for fold in range(1, CONFIG['n_folds'] + 1):
    print(f"\n--- Ekstraksi Fitur untuk Fold {fold} ---")

    # 1. Load Model Arsitektur untuk Ekstraksi Fitur (Tanpa Head Klasifikasi)
    feat_model = timm.create_model('swin_tiny_patch4_window7_224', pretrained=False, num_classes=0, global_pool='avg')

    # 2. Load Bobot Terbaik dari Training
    ckpt_path = os.path.join(ckpt_dir, f'fold_{fold}_best.pth')
    state_dict = torch.load(ckpt_path, map_location=device)

    # Filter bobot head klasifikasi (karena arsitektur feat_model tidak memilikinya)
    state_dict = {k: v for k, v in state_dict.items() if not k.startswith('head.')}
    feat_model.load_state_dict(state_dict, strict=False)
    feat_model.to(device)

    # 3. Load Data
    fold_dir = os.path.join(BASE_DRIVE, f'data/processed/fold_{fold}')
    df_train = pd.read_csv(os.path.join(fold_dir, 'train_paths.csv'))
    df_val = pd.read_csv(os.path.join(fold_dir, 'val_paths.csv'))
    df_test = pd.read_csv(os.path.join(fold_dir, 'test_paths.csv'))

    train_loader = DataLoader(FishEyeDataset(df_train, val_test_transform), batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'])
    val_loader = DataLoader(FishEyeDataset(df_val, val_test_transform), batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'])
    test_loader = DataLoader(FishEyeDataset(df_test, val_test_transform), batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'])

    # 4. Ekstraksi
    X_train_deep, y_train_deep = extract_features(feat_model, train_loader, device)
    X_val_deep, y_val_deep = extract_features(feat_model, val_loader, device)
    X_test_deep, y_test_deep = extract_features(feat_model, test_loader, device)

    # 5. Simpan ke Drive
    fold_save_dir = os.path.join(feat_dir, f'fold_{fold}')
    os.makedirs(fold_save_dir, exist_ok=True)

    np.save(os.path.join(fold_save_dir, 'train_deep.npy'), X_train_deep)
    np.save(os.path.join(fold_save_dir, 'val_deep.npy'), X_val_deep)
    np.save(os.path.join(fold_save_dir, 'test_deep.npy'), X_test_deep)

    np.save(os.path.join(fold_save_dir, 'train_labels.npy'), y_train_deep)
    np.save(os.path.join(fold_save_dir, 'val_labels.npy'), y_val_deep)
    np.save(os.path.join(fold_save_dir, 'test_labels.npy'), y_test_deep)

    print(f"  Shape Train: {X_train_deep.shape}, Val: {X_val_deep.shape}, Test: {X_test_deep.shape}")
    print(f"  Fitur deep Fold {fold} berhasil disimpan.")

print("\n" + "="*50)
print("NOTEBOOK 2 SELESAI. SELURUH FITUR DEEP BERHASIL DIEKSTRAKSI.")
print("="*50)


--- Ekstraksi Fitur untuk Fold 1 ---


  Shape Train: (2809, 768), Val: (703, 768), Test: (878, 768)
  Fitur deep Fold 1 berhasil disimpan.

--- Ekstraksi Fitur untuk Fold 2 ---


  Shape Train: (2809, 768), Val: (703, 768), Test: (878, 768)
  Fitur deep Fold 2 berhasil disimpan.

--- Ekstraksi Fitur untuk Fold 3 ---


  Shape Train: (2809, 768), Val: (703, 768), Test: (878, 768)
  Fitur deep Fold 3 berhasil disimpan.

--- Ekstraksi Fitur untuk Fold 4 ---


  Shape Train: (2809, 768), Val: (703, 768), Test: (878, 768)
  Fitur deep Fold 4 berhasil disimpan.

--- Ekstraksi Fitur untuk Fold 5 ---


  Shape Train: (2809, 768), Val: (703, 768), Test: (878, 768)
  Fitur deep Fold 5 berhasil disimpan.

NOTEBOOK 2 SELESAI. SELURUH FITUR DEEP BERHASIL DIEKSTRAKSI.
